# Binary Classification on `make_circles`: Linear Baseline vs. MLP

**Project 1 — OCI AI Foundations course exercise, extended with an actual evaluation.**

The `make_circles` dataset is two interlocking, non-linearly separable classes.
The point of this notebook is not just to draw a decision boundary — it's to
**prove**, with held-out test accuracy, that a linear model fails on this data
and a small neural network does not. We will train an MLP,
plotted a nice curved boundary, and stopped there. Below we add:

1. A train/test split (so "it works" means something on unseen data).
2. A linear baseline (`LogisticRegression`) fit on the *same* split, to show
   the failure the exercise claims exists.
3. A sweep over hidden layer size, recording test accuracy at each step —
   this is what the interactive slider below lets you explore by eye
   here as a saved, static result.
4. Saved artifacts: `results/metrics.json`, `results/accuracy_vs_hidden_size.png`,
   `results/decision_boundary.png`.


## Step 1: Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 0  # fixed seed used everywhere below for reproducibility


## Step 2: Generate data and split into train/test

We hold out 30% of the data for testing. Everything reported below is measured on this held-out set, not on training data.

In [ ]:
X, y = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


## Step 3: Linear baseline

This is the linear baseline model. We fit it and report its test accuracy to demonstrate its failure.

In [ ]:
baseline = LogisticRegression()
baseline.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))

print(f"Logistic Regression (linear baseline) test accuracy: {baseline_acc:.3f}")
print(classification_report(y_test, baseline.predict(X_test)))


## Step 4: MLP at a fixed hidden layer size

The MLP model, evaluated on held-out data.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', max_iter=3000, random_state=1)
mlp.fit(X_train, y_train)
mlp_acc = accuracy_score(y_test, mlp.predict(X_test))

print(f"MLP (hidden_layer_size=10) test accuracy: {mlp_acc:.3f}")
print(classification_report(y_test, mlp.predict(X_test)))
print("Confusion matrix:\n", confusion_matrix(y_test, mlp.predict(X_test)))


## Step 5: Does hidden layer size actually matter?

You can explore this visually with a slider below, but we will also record
what happened. Here we sweep hidden layer size, refit each time, and save the
result — this is the real experiment, not just a picture.

In [ ]:
hidden_sizes = [1, 2, 3, 5, 10, 20, 50]
sweep_results = []

for h in hidden_sizes:
    clf = MLPClassifier(hidden_layer_sizes=(h,), activation='relu', max_iter=3000, random_state=1)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    sweep_results.append({"hidden_layer_size": h, "test_accuracy": acc})
    print(f"hidden_layer_size={h:>3}  ->  test accuracy={acc:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sizes = [r["hidden_layer_size"] for r in sweep_results]
accs = [r["test_accuracy"] for r in sweep_results]
ax.plot(sizes, accs, marker='o', label='MLP')
ax.axhline(baseline_acc, color='red', linestyle='--', label=f'Logistic Regression baseline ({baseline_acc:.2f})')
ax.set_xlabel('Hidden layer size (neurons)')
ax.set_ylabel('Test accuracy')
ax.set_title('Test accuracy vs. hidden layer size (make_circles)')
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'accuracy_vs_hidden_size.png', dpi=150)
plt.show()


## Step 6: Decision boundary plot (best hidden size found above)

In [ ]:
best = max(sweep_results, key=lambda r: r["test_accuracy"])
best_h = best["hidden_layer_size"]

clf = MLPClassifier(hidden_layer_sizes=(best_h,), activation='relu', max_iter=3000, random_state=1)
clf.fit(X_train, y_train)

x_vals = np.linspace(X[:, 0].min() - 0.1, X[:, 0].max() + 0.1, 200)
y_vals = np.linspace(X[:, 1].min() - 0.1, X[:, 1].max() + 0.1, 200)
X_plane, Y_plane = np.meshgrid(x_vals, y_vals)
grid_points = np.column_stack((X_plane.ravel(), Y_plane.ravel()))
Z = clf.predict(grid_points).reshape(X_plane.shape)

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(X_plane, Y_plane, Z, alpha=0.3)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', label='train')
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolors='red', marker='s', label='test')
ax.set_title(f'Decision boundary, hidden_layer_size={best_h} (test acc={best["test_accuracy"]:.2f})')
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'decision_boundary.png', dpi=150)
plt.show()


## Step 7: Interactive exploration (optional, for live experimentation)

Interactive tool for use in Jupyter — not required to reproduce the results above, which are already saved as static artifacts.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interactive

def update_plot(hidden_layer_size):
    clf = MLPClassifier(hidden_layer_sizes=(hidden_layer_size,), activation='relu', max_iter=3000, random_state=1)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))

    Z = clf.predict(grid_points).reshape(X_plane.shape)
    plt.clf()
    plt.contourf(X_plane, Y_plane, Z, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k')
    plt.title(f'Hidden layer size: {hidden_layer_size}  |  test accuracy: {acc:.2f}')
    plt.show()

interactive_plot = interactive(update_plot, hidden_layer_size=widgets.IntSlider(min=1, max=50, step=1, value=10))
display(interactive_plot)


## Step 8: Save all metrics to disk

In [ ]:
metrics = {
    "dataset": "sklearn.datasets.make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=0)",
    "train_test_split": "70/30, stratified, random_state=0",
    "baseline_logistic_regression_test_accuracy": baseline_acc,
    "mlp_fixed_hidden10_test_accuracy": mlp_acc,
    "hidden_layer_sweep": sweep_results,
    "best_hidden_layer_size": best_h,
    "best_test_accuracy": best["test_accuracy"],
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
